# MANGO - Autonomous Colab GPU Worker

This runs the fair-FL R&D loop **automatically**: it pulls the latest code + job queue from
GitHub, runs the queued experiments on the GPU, and **pushes results back to the repo** so Claude
can read them and iterate - no copy/paste, no babysitting.

## One-time setup (required for push)
1. Create a GitHub **fine-grained Personal Access Token**: GitHub > Settings > Developer settings >
   Fine-grained tokens. Repository access = only `muzakkirhussain011/mango`. Permissions:
   **Contents = Read and write**. Copy the token.
2. In Colab, click the **key icon (Secrets)** in the left sidebar > **+ Add new secret**.
   Name = `GH_TOKEN`, Value = your token, and toggle **Notebook access ON**.
   **Do NOT paste the token into a cell or into chat.**
3. `Runtime > Change runtime type > GPU`. Then `Runtime > Run all`.

Leave this tab open. If Colab disconnects (idle / 12h cap / quota), just **Run all** again - the
worker skips finished jobs and resumes. To stop it, interrupt the last cell.

In [ ]:
import os, torch
print('CUDA:', torch.cuda.is_available(),
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (set Runtime > GPU)')
os.chdir('/content')
if not os.path.exists('/content/mango'):
    get_ipython().system('git clone -b main https://github.com/muzakkirhussain011/mango.git')
os.chdir('/content/mango')
get_ipython().system('git pull --ff-only')
get_ipython().system('pip install -e . -q')
print('setup done')

In [ ]:
import os, sys
os.chdir('/content/mango')
# Load the GitHub token from Colab Secrets (kernel context) into the environment.
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
    if tok:
        os.environ['GH_TOKEN'] = tok
        print('GH_TOKEN loaded from Colab secret (value hidden).')
    else:
        print('GH_TOKEN secret is empty - worker will run but cannot push.')
except Exception as e:
    print('No GH_TOKEN secret available - worker will run but cannot push. Details:', e)

sys.path.insert(0, '/content/mango/scripts')
import colab_worker
colab_worker.main()   # loops forever: runs queued jobs on GPU, pushes results, repeats